In [ ]:
# ===== 0. SETUP =====
import torch
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ===== 1. LOAD ALL FOLDS =====
from datasets import load_dataset

fold1 = load_dataset("RationAI/PanNuke", split="fold1")
fold2 = load_dataset("RationAI/PanNuke", split="fold2")
fold3 = load_dataset("RationAI/PanNuke", split="fold3")

print("Fold sizes:", len(fold1), len(fold2), len(fold3))

In [ ]:
# ===== 2. INSPECT SAMPLE =====
sample = fold1[0]   # just pick any fold for inspection

print("Keys:", sample.keys())
print("Image shape:", np.array(sample["image"]).shape)
print("Number of instances:", len(sample["instances"]))
print("Categories:", sample["categories"])
print("Tissue:", sample["tissue"])

In [ ]:
# ===== 3. VISUALIZE =====
img = np.array(sample["image"])

plt.imshow(img)
plt.title("Raw Image")
plt.axis("off")
plt.show()

In [ ]:
# ===== 4. INSTANCE → SEMANTIC =====

def build_semantic_mask(instances, categories, shape):
    H, W = shape
    mask = np.zeros((H, W), dtype=np.uint8)  # background = 0

    for inst, cat in zip(instances, categories):
        inst = np.array(inst)
        mask[(inst == 1) & (mask == 0)] = cat + 1  # shift so bg=0, classes=1..5

    return mask

In [ ]:
# ===== 5. CHECK MASK =====
sem_mask = build_semantic_mask(
    sample["instances"],
    sample["categories"],
    img.shape[:2]
)

print("Mask unique values:", np.unique(sem_mask))

plt.imshow(sem_mask, cmap="jet")
plt.title("Semantic Mask")
plt.colorbar()
plt.axis("off")
plt.show()

In [ ]:
# ===== 6. OVERLAY =====
plt.figure(figsize=(6,6))
plt.imshow(img)
plt.imshow(sem_mask, alpha=0.5, cmap="jet")
plt.title("Overlay")
plt.axis("off")
plt.show()

In [ ]:
# ===== 7. DATASET CLASS =====

class PanNukeSemanticDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        img = np.array(sample["image"]).astype(np.float32) / 255.0

        # ImageNet normalization
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])

        img = (img - mean) / std
        mask = build_semantic_mask(
            sample["instances"],
            sample["categories"],
            img.shape[:2]
        )

        # HWC → CHW
        img = np.transpose(img, (2, 0, 1))

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.long)
        )

In [ ]:
from datasets import concatenate_datasets

# ===== TRAIN/VAL SPLIT =====
train_data = concatenate_datasets([fold1, fold2])
val_data = fold3

print("Train size:", len(train_data))
print("Val size:", len(val_data))

In [ ]:
# datasets
train_ds = PanNukeSemanticDataset(train_data)
val_ds = PanNukeSemanticDataset(val_data)

# loaders
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

In [ ]:
# sanity batch
imgs, masks = next(iter(train_loader))
print("Mask values:", torch.unique(masks))
print("Batch images:", imgs.shape)
print("Batch masks:", masks.shape)

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
# ===== 9. MODEL =====
import segmentation_models_pytorch as smp

model = smp.MAnet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6,  # bg + 5 classes
    activation=None
)

model = model.to(device)
print("Model loaded")

In [ ]:
# ===== 10. LOSS =====
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss

dice_loss = DiceLoss(mode="multiclass")
focal_loss = FocalLoss(mode="multiclass")

def loss_fn(pred, target):
    return dice_loss(pred, target) + focal_loss(pred, target)

In [ ]:
# ===== DICE METRIC =====
def dice_score(preds, targets, num_classes=6):
    preds = torch.argmax(preds, dim=1)

    dice_per_class = []

    for cls in range(1, num_classes):  # skip background
        pred_cls = (preds == cls).float()
        target_cls = (targets == cls).float()

        intersection = (pred_cls * target_cls).sum()
        union = pred_cls.sum() + target_cls.sum()

        if union == 0:
            dice = torch.tensor(1.0)
        else:
            dice = (2. * intersection) / union

        dice_per_class.append(dice.item())

    return np.mean(dice_per_class), dice_per_class

In [ ]:
# ===== 11. OPTIMIZER =====
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# ===== 12. TRAIN LOOP =====

def train_one_epoch(loader):
    model.train()
    total_loss = 0

    for i, (imgs, masks) in enumerate(loader):
        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)

        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# ===== VALIDATION (LOSS + DICE) =====
def validate(loader):
    model.eval()
    total_loss = 0
    total_dice = 0

    class_dice_accum = np.zeros(5)

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss = loss_fn(preds, masks)

            total_loss += loss.item()

            mean_dice, class_dice = dice_score(preds, masks)
            total_dice += mean_dice
            class_dice_accum += np.array(class_dice)

    avg_loss = total_loss / len(loader)
    avg_dice = total_dice / len(loader)
    avg_class_dice = class_dice_accum / len(loader)

    return avg_loss, avg_dice, avg_class_dice

In [ ]:
# ===== TRAIN + EARLY STOPPING (DICE) =====
EPOCHS = 50
patience = 12

best_dice = 0
epochs_no_improve = 0

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(train_loader)
    val_loss, val_dice, class_dice = validate(val_loader)

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Dice: {val_dice:.4f}")
    print(f"Class Dice: {np.round(class_dice, 3)}")

    if val_dice > best_dice:
        best_dice = val_dice
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved best model (Dice improved)")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print("⛔ Early stopping")
        break

In [ ]:
# ===== 14. VISUALIZE PRED =====
model.eval()

imgs, masks = next(iter(train_loader))
imgs = imgs.to(device)

with torch.no_grad():
    preds = model(imgs)
    preds = torch.argmax(preds, dim=1)

img = imgs[0].cpu().permute(1,2,0).numpy()

# de-normalize
img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
img = np.clip(img, 0, 1)

gt = masks[0].numpy()
pred = preds[0].cpu().numpy()

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(gt, cmap="jet")
plt.title("GT")

plt.subplot(1,3,3)
plt.imshow(pred, cmap="jet")
plt.title("Prediction")

plt.show()

In [ ]:
# ===== VALIDATION VISUAL =====
model.eval()

imgs, masks = next(iter(val_loader))   # ← IMPORTANT: val_loader
imgs = imgs.to(device)

with torch.no_grad():
    preds = model(imgs)
    preds = torch.argmax(preds, dim=1)

img = imgs[0].cpu().permute(1,2,0).numpy()

# de-normalize
img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
img = np.clip(img, 0, 1)

gt = masks[0].numpy()
pred = preds[0].cpu().numpy()

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Val Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(gt, cmap="jet")
plt.title("Val GT")

plt.subplot(1,3,3)
plt.imshow(pred, cmap="jet")
plt.title("Val Prediction")

plt.show()